In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
TRAINING_DIR = DATA_DIR / "training"

df = pd.read_pickle(TRAINING_DIR / "candidate_job_cleaned.pkl")

print("Dataset shape:", df.shape)
print(df.head(2))

Dataset shape: (5000, 21)
                   pair_id  candidate_id               job_id pair_type  \
0  170_linkedin_4383910678           170  linkedin_4383910678  positive   
1  837_linkedin_4306118481           837  linkedin_4306118481  positive   

           candidate_roles                        candidate_skills_primary  \
0  [Junior HR Coordinator]  [Talent Acquisition, HRIS, Employee Relations]   
1            [ui engineer]    [Java, JavaScript, React, Git, Linux, Figma]   

  candidate_skills_secondary  candidate_experience_years candidate_seniority  \
0           [Workday, Excel]                           2              Junior   
1   [Sketch, Adobe XD, HTML]                           5                 Mid   

  candidate_domains  ...           job_title  \
0         [HR Tech]  ...           Recruiter   
1       [Education]  ...  Frontend Developer   

                                     job_description  \
0  Job Requisition ID #\n\n26WD95409\n\nPosition ...   
1  Skill Set Re

In [2]:
def combine_skills(row):
    primary = row["candidate_skills_primary"]
    secondary = row["candidate_skills_secondary"]

    if not isinstance(primary, list):
        primary = []

    if not isinstance(secondary, list):
        secondary = []

    return list(dict.fromkeys(primary + secondary))


df["candidate_all_skills"] = df.apply(combine_skills, axis=1)

df["candidate_skill_count"] = df["candidate_all_skills"].apply(len)

print(df["candidate_skill_count"].describe())

count    5000.000000
mean        7.159200
std         1.757178
min         2.000000
25%         6.000000
50%         7.000000
75%         9.000000
max        10.000000
Name: candidate_skill_count, dtype: float64


In [3]:
def create_candidate_text(row):
    roles = row["candidate_roles"]
    skills = row["candidate_all_skills"]
    domains = row["candidate_domains"]

    if not isinstance(roles, list):
        roles = []

    if not isinstance(domains, list):
        domains = []

    return (
        "Roles: " + ", ".join(map(str, roles)) +
        " | Skills: " + ", ".join(map(str, skills)) +
        " | Domains: " + ", ".join(map(str, domains)) +
        " | Career Intent: " + str(row["candidate_career_intent"])
    )


df["candidate_text"] = df.apply(create_candidate_text, axis=1)

df["job_text"] = (
    df["job_title"].fillna("").astype(str)
    + " "
    + df["job_description"].fillna("").astype(str)
)

print(df[["candidate_text", "job_text"]].head(2))

                                      candidate_text  \
0  Roles: Junior HR Coordinator | Skills: Talent ...   
1  Roles: ui engineer | Skills: Java, JavaScript,...   

                                            job_text  
0  Recruiter Job Requisition ID #\n\n26WD95409\n\...  
1  Frontend Developer Skill Set Required - \n\nMu...  


In [4]:
experience_map = {
    "Internship": 0,
    "Entry level": 1,
    "Associate": 2,
    "Mid-Senior level": 5,
    "Director": 8,
    "Executive": 10,
    "Not Applicable": 0
}

df["expected_experience_years"] = (
    df["job_seniority_level"]
    .map(experience_map)
    .fillna(0)
)

df["experience_gap"] = (
    df["candidate_experience_years"]
    - df["expected_experience_years"]
)

df["experience_fit"] = (
    1
    - df["experience_gap"].abs()
    / (df["expected_experience_years"] + 1)
)

df["experience_fit"] = df["experience_fit"].clip(0, 1)

print(
    df[
        [
            "candidate_experience_years",
            "expected_experience_years",
            "experience_gap",
            "experience_fit"
        ]
    ].head()
)

   candidate_experience_years  expected_experience_years  experience_gap  \
0                           2                          5              -3   
1                           5                          5               0   
2                           1                          1               0   
3                          12                          8               4   
4                           3                          5              -2   

   experience_fit  
0        0.500000  
1        1.000000  
2        1.000000  
3        0.555556  
4        0.666667  


In [5]:
skills = set()

for skill_list in df["candidate_all_skills"]:
    if isinstance(skill_list, list):
        for skill in skill_list:
            skills.add(str(skill).lower().strip())

print("Skill vocabulary:", len(skills))

Skill vocabulary: 782


In [6]:
def get_job_skills(text):
    text = str(text).lower()
    found = []

    for skill in skills:
        if skill and skill in text:
            found.append(skill)

    return set(found)


df["job_skills"] = df["job_text"].apply(get_job_skills)

df["job_skill_count"] = df["job_skills"].apply(len)

df["matched_skill_count"] = df.apply(
    lambda row: len(
        set(str(x).lower() for x in row["candidate_all_skills"])
        & row["job_skills"]
    ),
    axis=1
)

df["job_skill_coverage"] = (
    df["matched_skill_count"]
    / df["job_skill_count"].replace(0, np.nan)
).fillna(0)

df["candidate_skill_coverage"] = (
    df["matched_skill_count"]
    / df["candidate_skill_count"].replace(0, np.nan)
).fillna(0)

print(
    df[
        [
            "job_skill_count",
            "matched_skill_count",
            "job_skill_coverage",
            "candidate_skill_coverage"
        ]
    ].head()
)

   job_skill_count  matched_skill_count  job_skill_coverage  \
0                6                    0            0.000000   
1               14                    3            0.214286   
2               11                    1            0.090909   
3               28                    7            0.250000   
4               19                    3            0.157895   

   candidate_skill_coverage  
0                  0.000000  
1                  0.333333  
2                  0.166667  
3                  0.700000  
4                  0.750000  


In [7]:
def normalize_role(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+#.]+", " ", text)
    return set(text.split())


def role_match(row):
    roles = row["candidate_roles"]

    if not isinstance(roles, list):
        roles = []

    candidate_words = set()

    for role in roles:
        candidate_words.update(normalize_role(role))

    job_words = normalize_role(row["job_title"])

    if not candidate_words or not job_words:
        return 0

    common = candidate_words.intersection(job_words)

    return len(common) / len(job_words)


df["role_match_score"] = df.apply(role_match, axis=1)

print(df["role_match_score"].describe())

count    5000.000000
mean        0.137720
std         0.231243
min         0.000000
25%         0.000000
50%         0.000000
75%         0.250000
max         1.000000
Name: role_match_score, dtype: float64


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

candidate_embeddings = model.encode(
    df["candidate_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

job_embeddings = model.encode(
    df["job_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [9]:
df["semantic_similarity"] = np.sum(
    candidate_embeddings * job_embeddings,
    axis=1
) / (
    np.linalg.norm(candidate_embeddings, axis=1)
    * np.linalg.norm(job_embeddings, axis=1)
)

print(df["semantic_similarity"].describe())

count    5000.000000
mean        0.393404
std         0.131313
min         0.013743
25%         0.298504
50%         0.380684
75%         0.473856
max         0.771311
Name: semantic_similarity, dtype: float64


In [10]:
features = [
    "candidate_experience_years",
    "candidate_skill_count",
    "expected_experience_years",
    "experience_gap",
    "experience_fit",
    "job_skill_count",
    "matched_skill_count",
    "job_skill_coverage",
    "candidate_skill_coverage",
    "role_match_score",
    "job_remote_hint",
    "semantic_similarity"
]

X = df[features].copy()

y = df["match_label"].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures:")
print(features)

Feature shape: (5000, 12)
Target shape: (5000,)

Features:
['candidate_experience_years', 'candidate_skill_count', 'expected_experience_years', 'experience_gap', 'experience_fit', 'job_skill_count', 'matched_skill_count', 'job_skill_coverage', 'candidate_skill_coverage', 'role_match_score', 'job_remote_hint', 'semantic_similarity']


In [11]:
X.describe()

,candidate_experience_years,candidate_skill_count,expected_experience_years,experience_gap,experience_fit,job_skill_count,matched_skill_count,job_skill_coverage,candidate_skill_coverage,role_match_score,semantic_similarity
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,6.015600,7.159200,2.360600,3.655000,0.245186,16.195600,1.099800,0.067687,0.158477,0.137720,0.393404
std,3.762152,1.757178,2.638477,4.588964,0.353893,7.340308,1.474678,0.088384,0.211331,0.231243,0.131313
min,0.000000,2.000000,0.000000,-9.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.013743
25%,3.000000,6.000000,0.000000,0.000000,0.000000,11.000000,0.000000,0.000000,0.000000,0.000000,0.298504
50%,5.000000,7.000000,2.000000,3.000000,0.000000,16.000000,1.000000,0.038462,0.111111,0.000000,0.380684
75%,8.000000,9.000000,5.000000,7.000000,0.500000,21.000000,2.000000,0.111111,0.250000,0.250000,0.473856
max,20.000000,10.000000,10.000000,20.000000,1.000000,46.000000,9.000000,0.583333,1.000000,1.000000,0.771311


In [12]:
X.isnull().sum()

candidate_experience_years    0
candidate_skill_count         0
expected_experience_years     0
experience_gap                0
experience_fit                0
job_skill_count               0
matched_skill_count           0
job_skill_coverage            0
candidate_skill_coverage      0
role_match_score              0
job_remote_hint               0
semantic_similarity           0
dtype: int64

In [13]:
y.value_counts()

match_label
0    3500
1    1500
Name: count, dtype: int64

In [14]:
final_df = X.copy()

final_df["match_label"] = y

final_df["pair_id"] = df["pair_id"]
final_df["candidate_id"] = df["candidate_id"]
final_df["job_id"] = df["job_id"]

output_path = TRAINING_DIR / "candidate_job_features.csv"

final_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", final_df.shape)

Saved: c:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\training\candidate_job_features.csv
Shape: (5000, 16)
